## Retrieval Augmented Generation (RAG) for Central Bank Monetary Policy

### 0. Imports

In [294]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [295]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from lib.clean import clean_content
from lib.chunk import chunk_content
from lib.embed import load_embedder, embed_chunks, export_full_corpus_doc
from lib.evaluate import evaluate_retrieval, build_candidate_review, export_candidate_review_doc, gold_ids, retrieved_ids
from lib.extract import get_content
from lib.generate import generate_response
from lib.retrieve import retrieve_chunks, parse_query_date, hybrid_rank, tokenize_stem
from rank_bm25 import BM25Okapi

import numpy as np
import regex as re

import anthropic





In [296]:
client = anthropic.Anthropic()

### 1. Build Embedded Corpus

In [297]:

#set path for folder containing pdfs 
pdf_path = Path(r'C:\\DS_ML\\machine-learning-engineer\\monetary-policy-rag\\data\\chile_banco_central')

#get contents of each pdf
files = pdf_path.glob("*.pdf")
content = get_content(files)

#clean doc contents and divide into chunks
embedder = load_embedder()

chunked = {}
for doc in content.keys():
    cleaned = clean_content(content[doc])
    chunked[doc] = chunk_content(embedder, cleaned)

# #turn chunks from text into embeddings
corpus_embedded = embed_chunks(embedder, chunked)

#export the text of all chunks in the corpus to a Word doc
export_full_corpus_doc(corpus_embedded, filepath="chunk_review.docx")


Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 57 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 65 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (397 > 128). Running this sequence through the model will result in indexing errors


Saved 93 comunicados, 726 chunks to chunk_review.docx


### 2. Prepare Gold Queries

In [298]:
#list of gold queries, difficulty level of retrieval for query, and comunicados we expect to contain the relevant chunks 

gold_queries = [
    # # --- Irrelevant ---
    {"query": "What is the Federal Reserve's current interest rate?", "difficulty": "irrelevant", "expected_comunicados": []},
    {"query": "How does the European Central Bank set monetary policy?", "difficulty": "irrelevant", "expected_comunicados": []},
    {"query": "What is the history of the Chilean peso currency design?", "difficulty": "irrelevant", "expected_comunicados": []},

    # # # --- Easy (single meeting, single fact) ---
    {"query": "What was the policy interest rate in March 2016?", "difficulty": "easy", "expected_comunicados": ["2016_marzo"]},
    {"query": "Did the Council cut or raise the rate in April 2017?", "difficulty": "easy", "expected_comunicados": ["2017_abril"]},
    {"query": "What was the inflation rate reported in the December 2018 meeting?", "difficulty": "easy", "expected_comunicados": ["2018_diciembre"]},
    {"query": "What rate did the Council set in May 2020?", "difficulty": "easy", "expected_comunicados": ["2020_mayo"]},

    # # # --- Medium (single meeting, requires synthesis of context + decision) ---
    {"query": "What was the Council's reasoning for raising rates in January 2022?", "difficulty": "medium", "expected_comunicados": ["2022_enero"]},
    {"query": "What non-conventional measures did the Bank announce in March 2020?", "difficulty": "medium", 'expected_comunicados': ['2020_marzo_a', '2020_marzo_b']},
    {"query": "How did the Council describe labor market conditions in the June 2020 meeting?", "difficulty": "medium", "expected_comunicados": ["2020_junio"]},
    {"query": "What was said about copper prices in the February 2018 meeting?", "difficulty": "medium", "expected_comunicados": ["2018_febrero"]},

   # --- Hard (time-series, multi-meeting synthesis — this corpus's real strength) ---
    {"query": "How did the policy rate change from March 2020 through July 2020?", "difficulty": "hard", "expected_comunicados": ["2020_marzo_a", "2020_marzo_b", "2020_mayo", "2020_junio", "2020_julio"]},

    {"query": "When did the Council begin raising rates after the pandemic-era cuts, and why?", "difficulty": "hard", "expected_comunicados": ["2021_julio", "2021_agosto", "2021_diciembre", "2021_octubre"]},

    {"query": "How did the Council's tone on inflation expectations shift between 2020 and 2022?", "difficulty": "hard", "expected_comunicados": [
        "2020_diciembre", "2020_julio", "2020_junio", "2020_marzo_b",
        "2020_mayo", "2020_octubre", "2020_enero", "2020_septiembre",
        "2021_agosto", "2021_diciembre", "2021_enero", "2021_julio", "2021_junio",
        "2021_marzo", "2021_mayo", "2021_octubre",
        "2022_diciembre", "2022_enero", "2022_julio", "2022_junio",
        "2022_marzo", "2022_mayo", "2022_octubre", "2022_septiembre"
    ]},

    {"query": "What non-conventional pandemic support measures were introduced, then later withdrawn or maintained, across 2020 and 2021?", "difficulty": "hard", "expected_comunicados": ["2020_marzo_a", "2020_marzo_b", "2020_junio", "2021_marzo"]},
]

In [299]:
#translate gold queries into spanish for later use

def translate_query(client, query_en):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"Translate this to Spanish. Return ONLY the translation, no preamble:\n\n{query_en}"
        }]
    )
    return response.content[0].text.strip()

# client = anthropic.Anthropic() 
for item in gold_queries:
    item['query_es'] = translate_query(client, item['query'])

### 3. Define Gold Chunks


This is done by first conducting retrieval on the chunks to get a generous set of relevant chunks for each query.

The retrieval process uses hybrid ranking to choose the n most relevant chunks for each gold query.

Hybrid Ranking uses the combination of the cosine similarity score and the BM25 score to arrive at a ranking of all chunks from the set of expected comunicados for each query.

Cosine similarity scores each chunk based on semantic closeness between query and chunk as measured by the dot product of the respective normalized embeddings. 

BM25 scores each chunk based on the relevance of keywords common to both query and chunk. 

The process for calculating the BM25 score for a chunk is as follows:

1. For the query, the algorithm weights:
   a. Term Frequency (TF) - the number of times each word in the query appears in the candidate chunk
      * A higher TF suggests more relevance of the chunk to the query and drives the score for the chunk up. e.g. (month, liquidity) 
      * TF makes sure chunks dont score higher because they are longer and can accumulate more matches (e.g chunk is 50% longer than average chunk). 
        * A constant b penalizes the chunk based on its length.
      * TF also makes sure that multiple instances of the same term in a chunk do not inflate the TF score (e.g inflacion appears 10 times in chunk).
        * A constant k1 penalizes the chunk by reducing the amount added to the TF score for each additional instance.
   b. Inverse Document Frequency (IDF) - derived from the number of chunks in the entire corpus of chunks (in all comunicados) that contain each matching word
      * A word found in more chunks is more common and does not contribute as much to the relevance of the chunk to the query, so it gets a lower IDF and drives the score down (e.g. month)

2. The sum of the products of TF and IDF for each word in the query equals the BM25 score for the chunk.

$$\text{score}(q,d) = \sum_{t \in q} \underbrace{\text{IDF}(t)}_{\text{b}} \cdot \underbrace{\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}}_{\text{a}}$$

where $f(t,d)$ is the count of term $t$ in chunk $d$, $|d|$ is the chunk length, and $\text{avgdl}$ is the average chunk length in the corpus.

Once cosine similarity and BM25 scores are calculated, the candidate chunks are ranked in descending order separately by each score. Each chunk's cosine rank and BM25 rank are then combined via Reciprocal Rank Fusion — each rank contributes 1/(k + rank), and the two contributions are summed to give the chunk's hybrid score. Ranks are used rather than raw scores because cosine similarities and BM25 scores are on incompatible scales.

The top n chunks by hybrid score are the retrieved chunks for the query. n is chosen to be relatively large so as to not filter out relevant chunks. 

After retrieval, the smaller subset of retrieved hunks for each query is then manually reviewed to identify gold ids.


#### Prepare corpus for BM25 scoring

In [300]:
#tokenize the corpus (after stopword and stem) for BM25 scoring

tokenized_corpus = [tokenize_stem(chunk['text']) for chunk in corpus_embedded]

#create bm25 object with tokenized corpus for scoring candidate chunks based on text matching
bm25 = BM25Okapi(tokenized_corpus)

#create an index for each chunk in the corpus
key_to_idx = {
    chunk['name'] + "_" + str(chunk['chunk_id']): i
    for i, chunk in enumerate(corpus_embedded)
}

#### Retrieve candidate chunks for gold queries

In [301]:

#for each gold query, get top n retrieved chunks by hybrid ranking and add to query dict
candidate_review = build_candidate_review(embedder, gold_queries, corpus_embedded, bm25, key_to_idx, k_rrf=5)

#export candidates to Word doc for manual review of the candidate chunks for the final gold IDs.
export_candidate_review_doc(candidate_review, corpus_embedded, filepath="candidate_review.docx")



Saved 15 queries to candidate_review.docx


In [302]:
for c in candidate_review:
    c['confirmed_related_chunks'] = list(c['candidates'])


#### Remove irrelevant chunks

In [303]:
# dictionary of irrelevant chunks after manual review

removals = {"When did the Council begin raising rates after the pandemic-era cuts, and why?": [
        ("2021_julio", 1),      # vaccination/reopening context, no policy content
        ("2021_julio", 3),      # market rates + other central banks, not the TPM decision
        ("2021_agosto", 1),     # global central banks, copper/oil — not the Council
        ("2021_agosto", 3),     # fixed-income market reaction, not the decision
        ("2021_agosto", 4),     # IPSA, credit, Q2 GDP — activity data
        ("2021_diciembre", 3),  # credit transmission, not the decision
    ],}

import json 

def apply_removals(results, removals):
    for r in results:
        to_remove = {tuple(t) for t in removals.get(r["query"], [])}
        if not to_remove:
            continue

        # check against candidates, not confirmed_related_chunks:
        # candidates never change, so re-running this function is safe
        candidate_keys = {(c["name"], c["chunk_id"]) for c in r["candidates"]}
        missing = to_remove - candidate_keys
        if missing:
            raise KeyError(
                f"stale removals for '{r['query'][:50]}': {sorted(missing)}"
            )

        before = len(r["confirmed_related_chunks"])
        r["confirmed_related_chunks"] = [
            c for c in r["confirmed_related_chunks"]
            if (c["name"], c["chunk_id"]) not in to_remove
        ]
        print(f"{r['query'][:55]}: {before} -> {len(r['confirmed_related_chunks'])}")

    return results

gold_queries = apply_removals(candidate_review, removals)

with open("eval/gold_queries.json", "w") as f:
    json.dump(gold_queries, f, indent=2, ensure_ascii=False)

When did the Council begin raising rates after the pand: 12 -> 6


In [304]:
julio_0 = next(c for c in corpus_embedded
               if c["name"] == "2021_julio" and c["chunk_id"] == 0)

q = next(r for r in gold_queries if "begin raising rates" in r["query"])
q["confirmed_related_chunks"].append({
    "chunk_id": 0,
    "name": "2021_julio",
    "rrf_score": None,          # hand-added, not retriever-scored
    "text_preview": julio_0["text"][:500],
})

In [305]:
r = retrieved_ids(embedder, q["query"], q["query_es"], corpus_embedded, bm25,
                  key_to_idx, threshold=0.4, k=15, k_rrf=5,
                  rerank_n=None, config="plus_hybrid", use_date_filter=True)
print(len(r), len(gold_ids(q)), len(set(r) & gold_ids(q)))


********* CONFIG:  plus_hybrid *********

15 7 3


In [306]:
stale, no_candidates = {}, []
for r in candidate_review:
    to_remove = {tuple(t) for t in removals.get(r["query"], [])}
    if not to_remove:
        continue
    if "candidates" not in r:
        no_candidates.append(r["query"][:45])
        continue
    keys = {(c["name"], c["chunk_id"]) for c in r["candidates"]}
    missing = to_remove - keys
    if missing:
        stale[r["query"][:45]] = sorted(missing)

for q, m in stale.items():
    print(f"STALE  {q}: {m}")
for q in no_candidates:
    print(f"NO CANDIDATES KEY  {q}")
print(f"\n{len(stale)} stale, {len(no_candidates)} missing candidates, {len(removals)} removal lists total")


0 stale, 0 missing candidates, 1 removal lists total


In [307]:
gold_queries

[{'query': "What is the Federal Reserve's current interest rate?",
  'query_es': '¿Cuál es la tasa de interés actual de la Reserva Federal?',
  'difficulty': 'irrelevant',
  'expected_comunicados': [],
  'candidates': [],
  'confirmed_related_chunks': []},
 {'query': 'How does the European Central Bank set monetary policy?',
  'query_es': '¿Cómo establece el Banco Central Europeo la política monetaria?',
  'difficulty': 'irrelevant',
  'expected_comunicados': [],
  'candidates': [],
  'confirmed_related_chunks': []},
 {'query': 'What is the history of the Chilean peso currency design?',
  'query_es': '¿Cuál es la historia del diseño de la moneda peso chileno?',
  'difficulty': 'irrelevant',
  'expected_comunicados': [],
  'candidates': [],
  'confirmed_related_chunks': []},
 {'query': 'What was the policy interest rate in March 2016?',
  'query_es': '¿Cuál era la tasa de interés de política monetaria en marzo de 2016?',
  'difficulty': 'easy',
  'expected_comunicados': ['2016_marzo'],


### 4. Evaluate Retrieval

The steps for evalulation are the following:

1. Retrieve k chunks for each gold query
2. Calculate precision, recall and reciprocal rank when k chunks are retrieved
   * Precision is the share of retrieved chunks that are in the set of gold chunks
   * Recall is the share of gold chunks that were retreved
   * Reciprocal Rank equals  1 divided by the rank of the first retrieved chunk that is
     in the set of gold chunks — so 1.0 if the top hit is correct, 0.5 if the
     second is, and 0 if none of the k are. Averaged across queries this is MRR.
3. Average each metric across the 15 gold queries to get the headline numbers


In [308]:
# Sweep parameters for retrieval tuning.

# k         = number of chunks to retrieve
# threshold = minimum cosine similarity score for a chunk to be ranked/retrieved
# k_rrf     = damping constant in Reciprocal Rank Fusion — 1/(k_rrf + rank), applied
#             to the cosine and BM25 ranks before summing.
#             Small k_rrf spreads the values out, so being ranked near the top matters
#             a lot and one strong rank can carry a chunk on its own.
#             Large k_rrf clusters the values, so rank differences matter less and a
#             chunk needs to score well on BOTH rankers to win.

sweep = {'k':[5,10,15,20],
          'threshold':[0.1, 0.4, 0.6, 0.9],
          'k_rrf': [1, 5, 10, 30, 60],
          'rerank_n': [20, 30, 40, 50, 60, 70]
          }

# for k_rrf in weet['k_rrf']:
#     print(f"\n*********** k_rrf = {k_rrf} ********\n")
#     precisions, recalls, rrs = evaluate_retrieval(embedder, gold_queries, corpus_embedded, bm25, key_to_idx, k=10, threshold=0.4, k_rrf=k_rrf)

# Coordinate sweep, not a full grid: k_rrf swept with k=10, threshold=0.4 held fixed.
# Best k_rrf = 5 on precision and recall (RR was better at higher k_rrf)
import pandas as pd
params = {"k": 15, 
          "threshold": 0.4,
          "k_rrf": 5,
          "rerank_n": 30}

k = params["k"]


results = pd.DataFrame(columns=["config", "tier", "P", f"R@{k}", "MRR"])
cos_sim_only = evaluate_retrieval(embedder, gold_queries, corpus_embedded, bm25, key_to_idx, params,config="cos_sim_only", use_date_filter = False)
plus_date_filter = evaluate_retrieval(embedder, gold_queries, corpus_embedded, bm25, key_to_idx, params,config="plus_date_filter")
plus_hybrid = evaluate_retrieval(embedder, gold_queries, corpus_embedded, bm25, key_to_idx, params,config="plus_hybrid")
plus_rerank = evaluate_retrieval(embedder, gold_queries, corpus_embedded, bm25, key_to_idx, params,config="plus_rerank")
plus_rerank_blend = evaluate_retrieval(embedder, gold_queries, corpus_embedded, bm25, key_to_idx, params,config="plus_rerank_blend")



********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********


********* CONFIG:  cos_sim_only *********

Abstention accuracy: 0/3

********* CONFIG:  plus_date_filter *********


********* CONFIG:  plus_date_filter *********


********* CONFIG:  plus_date_filter *********


********* CONFIG:  plus_date_filter *********


********* CONFIG:  plus_date_filter *********


********* CONFIG:  plus_date_filter *********


********* CONFIG:  plus_da

In [309]:
results = pd.DataFrame()
for r in [cos_sim_only, plus_date_filter, plus_hybrid, plus_rerank, plus_rerank_blend]:
 results = pd.concat([results,r])

matrix = results.pivot(index="config", columns="tier")

matrix = matrix.round(3)

for m in [f"P@{k}", f"R@{k}", "MRR"]:
    print(m,"\n")
    print(results.pivot(index="config", columns="tier", values=m)
                 .reindex(columns=["easy", "medium", "hard"])
                 .reindex(["cos_sim_only", "plus_date_filter", "plus_hybrid", "plus_rerank", "plus_rerank_blend"])
                 .round(3), "\n")


P@15 

tier                easy  medium   hard
config                                 
cos_sim_only       0.000   0.050  0.150
plus_date_filter   0.838   0.588  0.550
plus_hybrid        0.838   0.588  0.600
plus_rerank        0.838   0.588  0.567
plus_rerank_blend  0.838   0.588  0.633 

R@15 

tier                easy  medium   hard
config                                 
cos_sim_only       0.000   0.208  0.197
plus_date_filter   0.917   0.917  0.445
plus_hybrid        0.917   0.917  0.472
plus_rerank        0.917   0.917  0.451
plus_rerank_blend  0.917   0.917  0.532 

MRR 

tier               easy  medium   hard
config                                
cos_sim_only        0.0   0.091  0.583
plus_date_filter    1.0   1.000  1.000
plus_hybrid         1.0   1.000  1.000
plus_rerank         1.0   1.000  1.000
plus_rerank_blend   1.0   1.000  1.000 



In [310]:
plus_hybrid

,config,tier,P@15,R@15,MRR
0,plus_hybrid,easy,0.8375,0.916667,1.0
1,plus_hybrid,medium,0.5875,0.916667,1.0
2,plus_hybrid,hard,0.6000,0.471726,1.0


In [311]:
plus_rerank

,config,tier,P@15,R@15,MRR
0,plus_rerank,easy,0.837500,0.916667,1.0
1,plus_rerank,medium,0.587500,0.916667,1.0
2,plus_rerank,hard,0.566667,0.450893,1.0


In [312]:
q = gold_queries[3]   # March 2016
r = retrieve_chunks(embedder, q["query"], q["query_es"], corpus_embedded, bm25,
                  key_to_idx, threshold=0.4, k=15, k_rrf=5,
                  rerank_n=None, config="plus_hybrid", use_date_filter=True)
print("retrieved:", len(r), r)
print("gold:", gold_ids(q))
print("overlap:", len(r) & len(gold_ids(q)))


********* CONFIG:  plus_hybrid *********

retrieved: 4 [{'chunk_id': 0, 'score': np.float32(0.5834944), 'name': '2016_marzo', 'comunicado_year': 2016, 'decoded': '<s> Jueves 17 de marzo de 2016. Reunión de Política Monetaria – Marzo 2016 En su reunión mensual de política monetaria, el Consejo del Banco Central de Chile acordó mantener la tasa de interés de política monetaria en 3,5%. En lo externo, los mercados financieros han mostrado mayor calma en las últimas semanas, aunque siguen presentes los riesgos de nuevos episodios de inestabilidad. Así, los premios por riesgo se han reducido, al mismo tiempo que los índices bursátiles y los precios de las materias primas han subido. Por otra parte , las proyecciones de crecimiento', 'bm25_score': np.float64(10.853683666921883), 'semantic_rank': 1, 'bm25_rank': 1, 'rrf': 0.3333333333333333}, {'chunk_id': 1, 'score': np.float32(0.53074837), 'name': '2016_marzo', 'comunicado_year': 2016, 'decoded': ', las proyecciones de crecimiento global y 

### 5. Generate Responses

In [313]:
#generate responses

#plausible query with answers in text
query = "What was the policy of the Chilean Central Bank in may 2020 after COVID"
query_2 = "How did the Council describe labor market conditions in the June 2020 meeting?"

#irrelevant query - no answers in text
query_3 = "what is the recession gap and how does it relate to the phillips curve?"

#nonsense query - no answers in text
query_4 = "how do number of coconuts influence divorce"

query_5 = " What is the history of the Chilean peso currency design?"

#hard query

query_7 = "When did the Council begin raising rates after the pandemic-era cuts, and why?"

query_8 = "How did the Council's tone on inflation expectations shift between 2020 and 2022?"

query_9 = "What non-conventional pandemic support measures were introduced, then later withdrawn or maintained, across 2020 and 2021?"

In [314]:
gold_queries[12]

{'query': 'When did the Council begin raising rates after the pandemic-era cuts, and why?',
 'query_es': '¿Cuándo comenzó el Consejo a subir las tasas después de los recortes de la era pandémica, y por qué?',
 'difficulty': 'hard',
 'expected_comunicados': ['2021_julio',
  '2021_agosto',
  '2021_diciembre',
  '2021_octubre'],
 'candidates': [{'chunk_id': 6,
   'name': '2021_diciembre',
   'rrf_score': 0.31,
   'text_preview': 'tenido un retroceso en lo más reciente, marcado en parte por una mayor preocupación por la inflación. La inflación anual siguió aumentando en los últimos meses, situándose en 6,7% en noviembre. Resalta el alza de los precios volátiles (10,5% anual a noviembre), especialmente de los combustibles, alimentos y servicios que se reactivaron después de la pandemia. La variación anual del IPC subyacente —que excluye los precios volátiles— se ubicó en 4,7% en noviembre, acorde con las proyecciones del IPoM de septiembre. Diversos indicadores'},
  {'chunk_id': 10,
   'nam

In [315]:
#Set Query, Translate, Retrieve k chunks using retrieval process
query = query_7
query_es = translate_query(client, query)


top_n1 = retrieve_chunks(embedder, query, query_es, corpus_embedded, bm25, key_to_idx, k=15, threshold=0.4, rerank_n = 30, k_rrf = 5, use_date_filter = False, config="cos_sim_only")
answer = generate_response(embedder, query, top_n1)
print(answer)

top_n2 = retrieve_chunks(embedder, query, query_es, corpus_embedded, bm25, key_to_idx, k=15, threshold=0.4, rerank_n = 30, k_rrf = 5, use_date_filter = True, config="plus_date_filter")
answer = generate_response(embedder, query, top_n2)
print(answer)

top_n3 = retrieve_chunks(embedder, query, query_es, corpus_embedded, bm25, key_to_idx, k=15, threshold=0.4, rerank_n = 30, k_rrf = 5, use_date_filter = True, config="plus_hybrid")
answer = generate_response(embedder, query, top_n3)
print(answer)

top_n4 = retrieve_chunks(embedder, query, query_es, corpus_embedded, bm25, key_to_idx, k=15, threshold=0.4, rerank_n = 30, k_rrf = 5, use_date_filter = True, config="plus_rerank")
answer = generate_response(embedder, query, top_n4)
print(answer)
top_n5 = retrieve_chunks(embedder, query, query_es, corpus_embedded, bm25, key_to_idx, k=15, threshold=0.4, rerank_n = 30, k_rrf = 5, use_date_filter = True, config="plus_rerank_blend")
answer = generate_response(embedder, query, top_n5)
print(answer)






********* CONFIG:  cos_sim_only *********

Based on the sources provided, I cannot pinpoint an exact date or explicit announcement stating "the Council began raising the policy rate on [date]." None of the excerpts include a direct statement such as "the Council decided to increase the MPR at this meeting." However, the sources do show the buildup of inflationary pressures during 2021 that would be consistent with a tightening cycle:

- By June 2021, "la inflación total subió hasta 3,8% anual" [Source: 2021_julio, chunk 9], driven mainly by fuel prices.
- By November 2021, inflation had risen further to 6,7% anual, with core inflation (IPC subyacente) at 4,7%, and volatile prices (fuel, food) up 10,5% [Source: 2021_diciembre, chunk 6].
- In October 2021, inflation expectations had risen across all horizons, with some near-term measures surpassing 6% for year-end and two-year-ahead expectations (EEE/EOF median) above the 3% target, compounded by significant peso depreciation [Source: 2